## Exploratory Data Analysis

### Import Libraries

In [1]:
import sys
import polars as pl
import polars.selectors as pol_sel

### Show Python & Library Versions

In [2]:
l = 8
r = 12

print("Python".rjust(l), ":", sys.version[0:6].ljust(r))
print("Polars".rjust(l), ":", pl.__version__.ljust(r))

  Python : 3.11.4      
  Polars : 1.12.0      


### Load CSV File into Polars DataFrame

In [3]:
df = pl.read_csv("data/tokenized_access_logs.csv")

df

product,category,log_date,log_month,log_hour,department,ip,url_link
str,str,str,str,i64,str,str,str
"""adidas Brazuca 2017 Official M…","""baseball & softball""","""9/1/2017 6:00""","""Sep""",6,"""fitness ""","""37.97.182.65""","""/department/fitness/category/b…"
"""The North Face Women's Recon B…","""hunting & shooting""","""9/1/2017 6:00""","""Sep""",6,"""fan shop ""","""206.56.112.1""","""/department/fan%20shop/categor…"
"""adidas Kids' RG III Mid Footba…","""featured shops""","""9/1/2017 6:00""","""Sep""",6,"""apparel ""","""215.143.180.0""","""/department/apparel/category/f…"
"""Under Armour Men's Compression…","""electronics""","""9/1/2017 6:00""","""Sep""",6,"""footwear ""","""206.56.112.1""","""/department/footwear/category/…"
"""Pelican Sunstream 100 Kayak""","""water sports""","""9/1/2017 6:01""","""Sep""",6,"""fan shop ""","""136.108.56.242""","""/department/fan%20shop/categor…"
…,…,…,…,…,…,…,…
"""Nike Men's Free TR 5.0 TB Trai…","""as seen on tv!""","""10/9/2017 21:21""","""Oct""",21,"""footwear ""","""93.166.57.36""","""/department/footwear/category/…"
"""Under Armour Hustle Storm Medi…","""fitness accessories""","""10/9/2017 21:21""","""Oct""",21,"""footwear ""","""126.175.2.58""","""/department/footwear/category/…"
"""Under Armour Hustle Storm Medi…","""fitness accessories""","""10/9/2017 21:22""","""Oct""",21,"""footwear ""","""201.210.19.242""","""/department/footwear/category/…"


### Display First Few Rows to Understand Structure of Data

In [4]:
print(df.head())

shape: (5, 8)
┌────────────┬────────────┬────────────┬───────────┬──────────┬────────────┬───────────┬───────────┐
│ product    ┆ category   ┆ log_date   ┆ log_month ┆ log_hour ┆ department ┆ ip        ┆ url_link  │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆ ---      ┆ ---        ┆ ---       ┆ ---       │
│ str        ┆ str        ┆ str        ┆ str       ┆ i64      ┆ str        ┆ str       ┆ str       │
╞════════════╪════════════╪════════════╪═══════════╪══════════╪════════════╪═══════════╪═══════════╡
│ adidas     ┆ baseball & ┆ 9/1/2017   ┆ Sep       ┆ 6        ┆ fitness    ┆ 37.97.182 ┆ /departme │
│ Brazuca    ┆ softball   ┆ 6:00       ┆           ┆          ┆            ┆ .65       ┆ nt/fitnes │
│ 2017       ┆            ┆            ┆           ┆          ┆            ┆           ┆ s/categor │
│ Official   ┆            ┆            ┆           ┆          ┆            ┆           ┆ y/b…      │
│ M…         ┆            ┆            ┆           ┆          ┆            ┆ 

### Retrieve Basic Information About DataFrame

In [5]:
print(df.shape)
print(df.dtypes)

(469977, 8)
[String, String, String, String, Int64, String, String, String]


### Display Summary Statistics for All Columns

In [6]:
summary = df.describe()
print(summary)

shape: (9, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ product   ┆ category  ┆ log_date  ┆ … ┆ log_hour  ┆ departmen ┆ ip        ┆ url_link │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ t         ┆ ---       ┆ ---      │
│ str       ┆ str       ┆ str       ┆ str       ┆   ┆ f64       ┆ ---       ┆ str       ┆ str      │
│           ┆           ┆           ┆           ┆   ┆           ┆ str       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 469977    ┆ 469977    ┆ 469977    ┆ … ┆ 469977.0  ┆ 469977    ┆ 469977    ┆ 469977   │
│ null_coun ┆ 0         ┆ 0         ┆ 0         ┆ … ┆ 0.0       ┆ 0         ┆ 0         ┆ 0        │
│ t         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ mean      ┆ null      ┆ null      ┆ null      ┆ … ┆ 14.591827 ┆ null      ┆

### Find Longest Text Length in Each Column

In [7]:
# Create an empty list to store max lengths for each string column
longest_text_lengths = []

# Loop through the columns to check for string columns
string_columns = [col for col in df.columns if df[col].dtype == pl.Utf8]

max_lengths = {}
for col in string_columns:
    max_length = df.select(pl.col(col).str.len_chars().max()).to_numpy()[0, 0]
    max_lengths[col] = max_length

df_max_lengths = pl.DataFrame(max_lengths)

df_max_lengths

product,category,log_date,log_month,department,ip,url_link
u32,u32,u32,u32,u32,u32,u32
45,20,16,3,9,15,130


### Retrieve Data Types of All Columns

In [8]:
print("Column data types:\n", df.dtypes)

Column data types:
 [String, String, String, String, Int64, String, String, String]


### Count Unique Values in Each Column

In [9]:
all_columns = [col for col in df.columns]

for col in all_columns:
    unique_counts = df[col].n_unique()
    print(f"Unique values in {col} :".rjust(48), f"{unique_counts}".ljust(6))

                      Unique values in product : 76    
                     Unique values in category : 33    
                     Unique values in log_date : 160815
                    Unique values in log_month : 5     
                     Unique values in log_hour : 24    
                   Unique values in department : 6     
                           Unique values in ip : 3340  
                     Unique values in url_link : 152   


### Check Distribution of Numerical Columns

In [10]:
numerical_cols = [item for item in all_columns if item not in string_columns]
numerical_cols = [item for item in numerical_cols if item not in ['id']]

numerical_cols

for col in numerical_cols:
    distribution = df.select(col).describe()
    print(col)
    print(distribution, '\n\n')

log_hour
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ log_hour  │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 469977.0  │
│ null_count ┆ 0.0       │
│ mean       ┆ 14.591827 │
│ std        ┆ 5.574014  │
│ min        ┆ 0.0       │
│ 25%        ┆ 10.0      │
│ 50%        ┆ 15.0      │
│ 75%        ┆ 20.0      │
│ max        ┆ 23.0      │
└────────────┴───────────┘ 




### List Unique Values for Select Columns/Features

In [11]:
cols_2_check = [
    "product",
    "category",
    "log_month",
    "log_hour",
    "department",
    "url_link"
]

for col in cols_2_check:
    unique_values = df[col].unique().sort().to_list()
    print(f"Column: {col} [{len(unique_values)}]")
    print(f"Unique values: {unique_values}\n")

Column: product [76]
Unique values: ['Bag Boy Beverage Holder', 'Bag Boy M330 Push Cart', 'Bridgestone e6 Straight Distance NFL Carolina', 'Bridgestone e6 Straight Distance NFL San Dieg', 'Bridgestone e6 Straight Distance NFL Tennesse', 'Bushnell Pro X7 Jolt Slope Rangefinder', 'Cleveland Golf Collegiate My Custom Wedge 588', "Cleveland Golf Women's 588 RTX CB Satin Chrom", 'Clicgear 8.0 Shoe Brush', 'Clicgear Rovic Cooler Bag', "Columbia Men's PFG Anchor Tough T-Shirt", "Diamondback Boys' Insight 24 Performance Hybr", "Diamondback Women's Serene Classic Comfort Bi", 'Field & Stream Sportsman 16 Gun Fire Safe', 'Fitbit The One Wireless Activity & Sleep Trac', 'Garmin Approach S3 Golf GPS Watch', 'Garmin Approach S4 Golf GPS Watch', 'Garmin Forerunner 910XT GPS Watch', 'Glove It Imperial Golf Towel', 'Glove It Urban Brick Golf Towel', "Glove It Women's Imperial Golf Glove", "Glove It Women's Mod Oval Golf Glove", "Hirzl Men's Hybrid Golf Glove", "Hirzl Women's Soffft Flex Golf Glove", "